In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torchvision
from tqdm import tqdm

%matplotlib inline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('running on', device)

SAVE_FIGURES = False  # set to True to write the figures used on the slides to ./figures
def savefig(name):
    if SAVE_FIGURES:
        os.makedirs('figures', exist_ok=True)
        plt.savefig(f'figures/{name}.png', dpi=150, bbox_inches='tight')

In [ ]:
%%sh
# Download the data - you need to do this only once
wget --no-verbose --output-document=image_cc_1.jpg https://github.com/chrirupp/cv_course/raw/main/data/image_cc_1.jpg
wget --no-verbose --output-document=image_me.jpg https://github.com/chrirupp/cv_course/raw/main/data/image_me.jpg

# Lecture 18 - 3D and World Models

3D from a single image: a feed-forward network predicts metric depth for every pixel; with the pinhole model (Lecture 14) we lift the image to a 3D point cloud and render it from new viewpoints. The holes that appear are exactly what a generative (world) model has to fill in.

Run this notebook on Google Colab (see the [readme](https://github.com/chrirupp/cv_course)). Cells that load large pretrained models are faster with a GPU runtime (*Edit > Notebook settings > T4 GPU*).

In [ ]:
def load_rgb(filename, width=768):
    image = cv2.cvtColor(cv2.imread(filename), cv2.COLOR_BGR2RGB)
    return cv2.resize(image, (width, int(round(image.shape[0] * width / image.shape[1]))), interpolation=cv2.INTER_AREA)

images = {'image_cc_1': load_rgb('image_cc_1.jpg'), 'image_me': load_rgb('image_me.jpg')}
# horizontal field of view: from the EXIF data for the Christ Church photo (24 mm, full frame), a guess for the other
fov = {'image_cc_1': 2 * np.degrees(np.arctan(18 / 24)), 'image_me': 60.0}

## Monocular metric depth

Depth Anything V2 is a ViT (DINOv2 backbone, Lecture 17) with a dense prediction head, trained on large amounts of (pseudo-)labelled images. This version predicts *metric* depth (in metres) for outdoor scenes. A single image has no absolute scale (Lecture 14), so the metres come from what the network has learned about the world: how big buildings and people usually are.

In [ ]:
from transformers import AutoImageProcessor, AutoModelForDepthEstimation
model_name = 'depth-anything/Depth-Anything-V2-Metric-Outdoor-Small-hf'
depth_processor = AutoImageProcessor.from_pretrained(model_name)
depth_model = AutoModelForDepthEstimation.from_pretrained(model_name).to(device).eval()

def predict_depth(image):
    inputs = depth_processor(images=image, return_tensors='pt').to(device)
    with torch.no_grad():
        depth = depth_model(**inputs).predicted_depth
    return F.interpolate(depth[:, None], size=image.shape[:2], mode='bicubic', align_corners=False)[0, 0].cpu().numpy()

depths = {name: predict_depth(im) for name, im in images.items()}
fig, axs = plt.subplots(2, 2, figsize=(16, 10))
for row, name in zip(axs, images):
    row[0].imshow(images[name])
    im = row[1].imshow(depths[name], cmap='turbo_r', vmin=0, vmax=np.percentile(depths[name], 98))
    plt.colorbar(im, ax=row[1], label='depth (m)', fraction=0.03)
    row[1].set_title(f'predicted depth: {np.percentile(depths[name], 2):.0f} m to {np.percentile(depths[name], 98):.0f} m')
    for ax in row:
        ax.axis('off')
savefig('3d_depth')
plt.show()

## Lifting the image to 3D

Pixel $(u, v)$ with depth $z$ is the 3D point $\mathbf{X} = z\, K^{-1} (u, v, 1)^\top$ in camera coordinates: the inverse of the projection from Lecture 14. Then we move the camera and project the coloured points again. We render with a z-buffer: if two points land on the same pixel, the nearer one wins.

In [ ]:
def intrinsics(image, fov_degrees):
    h, w = image.shape[:2]
    f = w / 2 / np.tan(np.radians(fov_degrees) / 2)
    return np.array([[f, 0, w / 2], [0, f, h / 2], [0, 0, 1]])

def unproject(depth, K):
    h, w = depth.shape
    u, v = np.meshgrid(np.arange(w) + 0.5, np.arange(h) + 0.5)
    rays = np.stack([u, v, np.ones_like(u)], -1).reshape(-1, 3) @ np.linalg.inv(K).T
    return rays * depth.reshape(-1, 1)

def render(points, colors, K, R, t, shape, splat=2):
    """z-buffer rendering of a coloured point cloud with square splats; empty pixels are black"""
    h, w = shape
    Xc = points @ R.T + t
    front = Xc[:, 2] > 0.1
    Xc, c = Xc[front], colors[front]
    uv = Xc @ K.T
    uv = uv[:, :2] / uv[:, 2:]
    order = np.argsort(-Xc[:, 2])                 # far to near: nearer points overwrite farther ones
    uv, c, z = uv[order], c[order], Xc[order, 2]
    out = np.zeros((h, w, 3), np.uint8)
    hit = np.zeros((h, w), bool)
    for dy in range(splat):
        for dx in range(splat):
            x, y = np.floor(uv[:, 0]).astype(int) + dx, np.floor(uv[:, 1]).astype(int) + dy
            ok = (x >= 0) & (x < w) & (y >= 0) & (y < h)
            out[y[ok], x[ok]] = c[ok]
            hit[y[ok], x[ok]] = True
    return out, hit

def orbit(angle_degrees, center_depth, height=0.0, forward=0.0):
    """camera rotated by `angle` around a vertical axis through the point at `center_depth` in front of the original camera"""
    a = np.radians(angle_degrees)
    R = np.array([[np.cos(a), 0, -np.sin(a)], [0, 1, 0], [np.sin(a), 0, np.cos(a)]])
    pivot = np.array([0, 0, center_depth])
    t = -R @ pivot + pivot + np.array([0, height, -forward])
    return R, t

clouds = {}
for name, image in images.items():
    K = intrinsics(image, fov[name])
    clouds[name] = (unproject(depths[name], K), image.reshape(-1, 3), K)

# holes (pixels no point lands on) are shown in magenta: regions the camera never saw
for name, views in [('image_me', [('original view', 0, 0), ('orbit 15 degrees left', -15, 0), ('orbit 15 degrees right', 15, 0), ('5 m forward', 0, 5)]),
                    ('image_cc_1', [('original view', 0, 0), ('orbit 10 degrees left', -10, 0), ('orbit 10 degrees right', 10, 0), ('15 m forward', 0, 15)])]:
    points, colors, K = clouds[name]
    center = np.median(depths[name])
    fig, axs = plt.subplots(2, 2, figsize=(18, 12))
    for ax, (title, angle, forward) in zip(axs.ravel(), views):
        R, t = orbit(angle, center, forward=forward)
        rendered, hit = render(points, colors, K, R, t, images[name].shape[:2])
        rendered[~hit] = [255, 0, 255]
        ax.imshow(rendered)
        ax.set_title(f'{title}: {100 * (1 - hit.mean()):.0f}% of the pixels are holes')
        ax.axis('off')
    savefig(f'3d_novel_views_{name}')
    plt.show()

In [ ]:
# a side view shows the 3D structure: we only get the visible surface (2.5D), nothing behind it.
# Note the sky: it has no depth, the network puts it on a far 'dome'.
fig, axs = plt.subplots(1, 2, figsize=(18, 6))
for ax, name in zip(axs, images):
    points, colors, K = clouds[name]
    keep = np.random.default_rng(0).choice(len(points), 60000, replace=False)
    p, c = points[keep], colors[keep]
    near = p[:, 2] < np.percentile(p[:, 2], 95)
    ax.scatter(p[near, 2], -p[near, 1], c=c[near] / 255, s=0.5)
    ax.plot(0, 0, 'k>', markersize=12)
    ax.set_xlabel('depth z (m)')
    ax.set_ylabel('height (m)')
    ax.set_aspect('equal')
    ax.set_title(f'{name}: side view of the point cloud (camera on the left)')
savefig('3d_side_view')
plt.show()

In [ ]:
# fly around the scene
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
name = 'image_me'
points, colors, K = clouds[name]
center = np.median(depths[name])
angles = 12 * np.sin(np.linspace(0, 2 * np.pi, 48, endpoint=False))
fig, ax = plt.subplots(figsize=(8, 5.5), dpi=72)
im = ax.imshow(images[name])
ax.axis('off')
def update(i):
    R, t = orbit(angles[i], center)
    im.set_data(render(points, colors, K, R, t, images[name].shape[:2])[0])
anim = FuncAnimation(fig, update, frames=len(angles), interval=80)
plt.close()
HTML(anim.to_jshtml())